# Análisis exploratorio de datos

Este capítulo tiene el EDA completo en Python. Después de cada tabla o gráfico hay una caja verde con la interpretación. Las cajas naranjas marcan problemas de los datos que cambian la lectura de un resultado.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

pio.renderers.default = "notebook_connected"
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (9, 6), "figure.dpi": 110, "axes.titlesize": 14, "axes.titleweight": "bold",
                     "font.size": 11, "axes.edgecolor": "#BBBBBB"})
pd.set_option("display.max_rows", 250)

df = pd.read_csv("dataset_calidad_aire.csv")

In [ ]:
# Columnas que vienen como texto pero son números (tienen comas de miles)
df["Año"] = df["Año"].astype(str).str.replace(",", "").astype(int)
columnas_numericas = ["ID Estacion", "Suma", "No. de datos", "Promedio", "Mediana", "Percentil 98", "Máximo", "Mínimo"]
for col in columnas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(",", "").str.strip(), errors="coerce")

In [ ]:
# Tablas con formato y mapas con fondo satelital

def _num(x, k):
    if pd.isna(x):
        return "—"
    if isinstance(x, (int, np.integer)):
        return f"{x:,}".replace(",", ".") if abs(x) >= 10000 else str(x)
    return f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def tabla(d, columnas=None, decimales=2, indice=False):
    d = d.copy()
    if columnas is not None:
        d.columns = columnas
    formatos = {c: (lambda x, k=decimales: _num(x, k)) for c in d.select_dtypes("number").columns}
    estilo = d.style.format(formatos)
    if not indice:
        estilo = estilo.hide(axis="index")
    return estilo.set_table_styles([
        {"selector": "", "props": [("border-collapse", "collapse"), ("font-size", "0.88em"), ("margin", "0.4em 0 1.2em")]},
        {"selector": "thead th", "props": [("background", "#1c2b33"), ("color", "white"), ("font-weight", "500"),
                                           ("padding", "7px 12px"), ("text-align", "left")]},
        {"selector": "tbody td", "props": [("padding", "6px 12px"), ("border-bottom", "1px solid #d5dee2")]},
        {"selector": "tbody tr:nth-child(even) td", "props": [("background", "#f3f7f8")]},
    ])


ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"


def mapa_satelital(lat, lon, tamano, texto, titulo, color="#ffd23f", escala=None, titulo_barra=None):
    marcador = dict(size=tamano, color=color, opacity=0.85)
    if escala is not None:
        marcador.update(colorscale=escala, cmin=0, cmax=100, colorbar=dict(title=titulo_barra, thickness=12, len=0.6))
    fig = go.Figure(go.Scattermapbox(lat=lat, lon=lon, mode="markers", marker=marcador, text=texto, hoverinfo="text"))
    fig.update_layout(title=dict(text=titulo, x=0.01, font=dict(size=15)), height=620, margin=dict(l=0, r=0, t=45, b=0),
                      mapbox=dict(style="white-bg", zoom=4.4, center=dict(lat=4.6, lon=-74.2),
                                  layers=[dict(below="traces", sourcetype="raster", source=[ESRI],
                                               sourceattribution="Imágenes: Esri World Imagery")]))
    fig.show()

## 1. Preparación y calidad de los datos

In [ ]:
tabla(pd.DataFrame({"Filas": [df.shape[0]], "Columnas": [df.shape[1]]}))

```{admonition} Interpretación
:class: tip
El archivo tiene 29.683 filas y 28 columnas. Cada fila es el resumen de un año para una variable, una estación y un tiempo de exposición.
```

In [ ]:
estructura = pd.DataFrame({
    "Columna": df.columns,
    "Tipo": df.dtypes.astype(str).values,
    "Valores no nulos": df.notna().sum().values,
    "Valores nulos": df.isna().sum().values,
    "Ejemplo": [str(df[c].dropna().iloc[0])[:40] for c in df.columns],
})
tabla(estructura)

```{admonition} Interpretación
:class: tip
La tabla resume las 28 columnas. Después de la limpieza las columnas de medición quedan como números (float64 o int64) y las de texto son el nombre de la estación, la variable, las unidades, el departamento, el municipio y las fechas del máximo y el mínimo.
```

In [ ]:
faltantes = pd.DataFrame({"Columna": df.columns, "Nulos": df.isna().sum().values,
                          "Porcentaje": df.isna().mean().values * 100})
tabla(faltantes[faltantes["Nulos"] > 0], ["Columna", "Valores nulos", "Porcentaje (%)"], decimales=3)

```{admonition} Interpretación
:class: tip
Solo tres columnas tienen valores nulos y ninguna pasa del 0,5 %. Representatividad Temporal tiene 148, Código del Municipio 4 y Tipo de Estación 2. Las columnas que usa el análisis (Variable, Año y Promedio) están completas, así que no hubo que imputar ni borrar filas por esta razón.
```

In [ ]:
duplicados = pd.DataFrame({
    "Concepto": ["Filas en el archivo", "Filas únicas", "Filas duplicadas", "Porcentaje duplicado (%)"],
    "Valor": [len(df), len(df.drop_duplicates()), df.duplicated().sum(), round(df.duplicated().mean() * 100, 2)],
})
tabla(duplicados)

```{admonition} Interpretación
:class: tip
Casi la mitad del archivo (46,96 %) son filas repetidas. Parece que el consolidado se armó uniendo reportes que se solapaban. Si no se quitan, esas estaciones contarían doble en los promedios, así que desde aquí se trabaja con las 15.745 filas únicas.
```

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print("Filas después de quitar duplicados:", len(df))

### 1.1 Exploración general del dataset

In [ ]:
conteo_variables = df["Variable"].value_counts().sort_index().rename_axis("Variable").reset_index(name="Registros")
tabla(conteo_variables)

```{admonition} Interpretación
:class: tip
Hay 20 variables. PM10 es la que más registros tiene (2.984) y RUVb la que menos (66).
```

In [ ]:
print("Número de variables:", df["Variable"].nunique())

```{admonition} Interpretación
:class: tip
Las 20 variables se dividen en contaminantes (PM10, PM2.5, PST, NO, NO₂, O₃, SO₂, CO) y meteorológicas (temperatura, humedad, viento, presión, precipitación y radiación). La temperatura y la humedad vienen en tres versiones: sin altura, a 2 m y a 10 m. Son variables distintas y no se mezclan.
```

#### 1.1.1 Distribución de registros por estación

In [ ]:
print("Número de estaciones:", df["Estación"].nunique())

In [ ]:
registros_estacion = df["Estación"].value_counts().rename_axis("Estación").reset_index(name="n")
tabla(registros_estacion.head(10), ["Estación", "Registros"])

In [ ]:
def barras_h(d, cat, val, titulo, xlab, ylab, subtitulo=None, alto=6, tam=8, total=None, porcentaje=True, dec=0):
    """Barras horizontales. Si porcentaje=True la etiqueta muestra N arriba y (%) debajo."""
    d = d.sort_values(val)
    tot = total if total is not None else d[val].sum()
    fig, ax = plt.subplots(figsize=(9, alto))
    ax.barh(d[cat].astype(str), d[val], color="#4a6572")
    for y, v in zip(d[cat].astype(str), d[val]):
        if porcentaje:
            etiqueta = f"{v:,.0f}".replace(",", ".") + f"\n({v / tot * 100:.0f} %)"
        else:
            etiqueta = f"{v:.{dec}f}".replace(".", ",")
        ax.text(v + d[val].max() * 0.01, y, etiqueta, va="center", fontsize=tam, linespacing=0.95)
    ax.set_xlim(0, d[val].max() * 1.16)
    ax.set_title(titulo if subtitulo is None else titulo + "\n" + subtitulo, fontweight="bold")
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.grid(axis="y", visible=False)
    plt.tight_layout(); plt.show()

In [ ]:
top_estaciones = registros_estacion.nlargest(15, "n")
barras_h(top_estaciones, "Estación", "n", "15 estaciones con mayor número de registros", "Número de registros", "Estación",
         total=len(df), alto=7)

```{admonition} Interpretación
:class: tip
Las seis estaciones con más registros son de Bogotá (Centro de Alto Rendimiento, Kennedy, Tunal, Las Ferias, Guaymaral y Puente Aranda). Miden muchas variables durante todo el periodo, y por eso cada una tiene cerca del 2 % del total.
```

#### 1.1.2 Distribución de registros por variable ambiental

In [ ]:
frecuencia_variables = df["Variable"].value_counts().rename_axis("Variable").reset_index(name="n")
tabla(frecuencia_variables, ["Variable", "Registros"])

In [ ]:
barras_h(frecuencia_variables, "Variable", "n", "Número de registros por variable ambiental", "Número de registros",
         "Variable ambiental", alto=8)

```{admonition} Interpretación
:class: tip
PM10 (19 %) y PM2.5 (13 %) suman casi un tercio de los registros, porque el material particulado es lo que más se vigila. TAire, HAire y RUVb tienen menos de 80 registros cada una, así que lo que se diga de ellas se basa en muy pocos datos.
```

### 1.2 Distribución temporal de los registros

In [ ]:
frecuencia_anios = df["Año"].value_counts().rename_axis("Año").reset_index(name="n").sort_values("Año")
tabla(frecuencia_anios, ["Año", "Registros"])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(frecuencia_anios["Año"].astype(str), frecuencia_anios["n"], color="#4a6572")
total = frecuencia_anios["n"].sum()
for x, v in zip(frecuencia_anios["Año"].astype(str), frecuencia_anios["n"]):
    ax.text(x, v + 20, f"{v:,}".replace(",", ".") + f"\n({v / total * 100:.0f} %)", ha="center", fontsize=8, linespacing=0.95)
ax.set_ylim(0, frecuencia_anios["n"].max() * 1.16)
ax.set_title("Número de registros por año", fontweight="bold"); ax.set_xlabel("Año"); ax.set_ylabel("Número de registros")
plt.xticks(rotation=45, ha="right"); ax.grid(axis="x", visible=False); plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
Los registros pasan de 647 en 2011 (4 %) a 1.941 en 2022 (12 %), o sea que la red casi se triplicó. En 2023 y 2024 bajan otra vez. Como la red cambia tanto, un promedio nacional puede subir o bajar solo porque entraron o salieron estaciones.
```

### 1.3 Estadística descriptiva y distribución de variables ambientales

In [ ]:
estadistica_variables = df.groupby("Variable", as_index=False).agg(
    n=("Promedio", "size"), Media=("Promedio", "mean"), Mediana=("Promedio", "median"), Desv=("Promedio", "std"),
    Minimo=("Promedio", "min"), Q1=("Promedio", lambda s: s.quantile(0.25)), Q3=("Promedio", lambda s: s.quantile(0.75)),
    Maximo=("Promedio", "max"))
tabla(estadistica_variables, ["Variable", "n", "Media", "Mediana", "Desv. estándar", "Mínimo", "Q1", "Q3", "Máximo"])

```{admonition} Interpretación
:class: tip
Cada variable tiene su propia escala (la presión está cerca de 700 mmHg y el viento cerca de 1 m/s), así que solo se comparan consigo mismas.

En casi todos los contaminantes la media es mayor que la mediana, por ejemplo PM10 (37,1 y 34,6) y SO₂ (9,4 y 6,0). Unas pocas estaciones con valores altos suben el promedio. El caso más extremo es PLiquida, con una media de 9,8 mm y una mediana de 1,4 mm.
```

#### Distribución de las variables ambientales

In [ ]:
vars_orden = sorted(df["Variable"].unique())
def panel_boxplots(inicio, fin, subtitulo):
    grupo = vars_orden[inicio:fin]
    filas = int(np.ceil(len(grupo) / 2))
    fig, axes = plt.subplots(filas, 2, figsize=(9, 3.2 * filas))
    axes = np.array(axes).ravel()
    for ax, v in zip(axes, grupo):
        ax.boxplot(df.loc[df["Variable"] == v, "Promedio"].dropna(), widths=0.5)
        ax.set_title(v, fontweight="bold", fontsize=12); ax.set_xticks([]); ax.set_ylabel("Valor promedio"); ax.grid(axis="x", visible=False)
    for ax in axes[len(grupo):]: ax.axis("off")
    fig.suptitle("Distribución de las variables ambientales\n" + subtitulo, fontweight="bold", fontsize=15)
    plt.tight_layout(); plt.show()

In [ ]:
panel_boxplots(0, 5, "Variables ambientales 1-5")

In [ ]:
panel_boxplots(5, 10, "Variables ambientales 6-10")

In [ ]:
panel_boxplots(10, 15, "Variables ambientales 11-15")

In [ ]:
panel_boxplots(15, 20, "Variables ambientales 16-20")

```{admonition} Interpretación
:class: tip
En la mayoría de las variables los puntos sueltos están por encima de la caja, lo que confirma la cola hacia la derecha. La caja de RUVb es muy grande porque mezcla dos escalas de medición.
```

#### Identificación de valores atípicos

Se usa el criterio de Tukey: es atípico todo valor menor que Q1 − 1,5·IQR o mayor que Q3 + 1,5·IQR.

In [ ]:
outliers_variables = df.groupby("Variable", as_index=False).agg(Q1=("Promedio", lambda s: s.quantile(0.25)), Q3=("Promedio", lambda s: s.quantile(0.75)))
outliers_variables["IQR"] = outliers_variables["Q3"] - outliers_variables["Q1"]
outliers_variables["Limite_inferior"] = outliers_variables["Q1"] - 1.5 * outliers_variables["IQR"]
outliers_variables["Limite_superior"] = outliers_variables["Q3"] + 1.5 * outliers_variables["IQR"]

#### Resumen de valores atípicos

In [ ]:
def contar_atipicos(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    return ((s < q1 - 1.5 * (q3 - q1)) | (s > q3 + 1.5 * (q3 - q1))).sum()

outliers_variables["Atipicos"] = outliers_variables["Variable"].map(df.groupby("Variable")["Promedio"].apply(contar_atipicos))
outliers_variables["Porcentaje"] = outliers_variables["Atipicos"] / outliers_variables["Variable"].map(df["Variable"].value_counts()) * 100
print("Total de registros atípicos:", outliers_variables["Atipicos"].sum(), "de", len(df))
tabla(outliers_variables.sort_values("Porcentaje", ascending=False),
      ["Variable", "Q1", "Q3", "IQR", "Límite inferior", "Límite superior", "Atípicos", "% atípicos"])

```{admonition} Interpretación
:class: tip
En total hay 638 registros atípicos, el 4,1 % del dataset. La variable con más atípicos en proporción es HAire, con 12 de 75 registros (16 %) fuera de los límites de 40,6 y 97,8 %. Le siguen el SO₂, con 93 registros (8,7 %) por encima de 22,4 µg/m³, y el CO, con 62 registros (8,6 %) por encima de 1.666,1.

En el material particulado el límite superior es 71,5 µg/m³ para el PM10 (91 atípicos, 3,0 %) y 30,75 µg/m³ para el PM2.5 (74 atípicos, 3,7 %). El NO₂ solo tiene 1 atípico por encima de 64,25 µg/m³, y DViento y RUVb no tienen ninguno porque su rango intercuartílico es muy amplio (164,65 y 203,95).

Cuando el límite inferior es negativo, como en el CO (−224,31), el NO₂ (−12,15) o el SO₂ (−8,0), no puede haber atípicos bajos porque esas variables no toman valores negativos.
```

### 1.4 Representatividad temporal de los datos

In [ ]:
representatividad_variables = df.groupby("Variable", as_index=False).agg(Media=("Representatividad Temporal", "mean"), Mediana=("Representatividad Temporal", "median"), Minimo=("Representatividad Temporal", "min"), Maximo=("Representatividad Temporal", "max")).sort_values("Media", ascending=False)
tabla(representatividad_variables)

In [ ]:
barras_h(representatividad_variables, "Variable", "Media", "Representatividad temporal promedio por variable",
         "Representatividad temporal (%)", "Variable ambiental", alto=8, porcentaje=False, dec=1)

```{admonition} Interpretación
:class: tip
La mayoría de variables tiene entre 63 % y 75 % de representatividad, o sea que en un registro normal falta cerca de un tercio del año. TAire (24 %) y HAire (27 %) son las más bajas.
```

In [ ]:
fuera_de_rango = df[df["Representatividad Temporal"] > 100]
tabla(pd.DataFrame({"Registros con más de 100 %": [len(fuera_de_rango)],
                    "Máximo reportado (%)": [df["Representatividad Temporal"].max()]}), decimales=1)

```{admonition} Precaución
:class: warning
Una representatividad de más de 100 % no es posible, porque no se puede medir más que el año completo. Son errores de captura. Se dejan anotados pero no se corrigen porque no se sabe cuál es el valor real.
```

### 1.5 Distribución geográfica de las estaciones

In [ ]:
ubicacion_estaciones = (df.groupby("Estación").agg(Latitud=("Latitud", "mean"), Longitud=("Longitud", "mean"),
                                                 Municipio=("Nombre del Municipio", "first"), Registros=("Variable", "size"))
                        .reset_index())
print("Estaciones:", len(ubicacion_estaciones))
mapa_satelital(ubicacion_estaciones["Latitud"], ubicacion_estaciones["Longitud"], 8,
               ubicacion_estaciones["Estación"] + " (" + ubicacion_estaciones["Municipio"] + ")",
               "Ubicación de las estaciones de monitoreo", color="#7fdcff")

```{admonition} Interpretación
:class: tip
Las estaciones se concentran en la región Andina, sobre todo en Bogotá, el Valle de Aburrá y Cali. En el Caribe hay estaciones en Barranquilla, Santa Marta, Cartagena y en las zonas carboníferas del Cesar y La Guajira. La Amazonía, la Orinoquía y el Pacífico casi no tienen estaciones.
```

#### Distribución por tipo de estación

In [ ]:
frecuencia_tipo_estacion = df["Tipo de Estación"].value_counts().rename_axis("Tipo de Estación").reset_index(name="n")
frecuencia_tipo_estacion["Porcentaje"] = frecuencia_tipo_estacion["n"] / frecuencia_tipo_estacion["n"].sum() * 100
tabla(frecuencia_tipo_estacion, ["Tipo de estación", "Registros", "Porcentaje (%)"], decimales=1)

In [ ]:
barras_h(frecuencia_tipo_estacion, "Tipo de Estación", "n", "Distribución de registros por tipo de estación",
         "Número de registros", "Tipo de estación", alto=3.5)

```{admonition} Interpretación
:class: tip
El 90 % de los registros (14.184) viene de estaciones fijas, que miden todo el año. Las indicativas son campañas temporales y aportan el 10 % (1.560).
```

#### Distribución por departamento

In [ ]:
frecuencia_departamentos = df["Nombre del Departamento"].value_counts().rename_axis("Nombre del Departamento").reset_index(name="n")
tabla(frecuencia_departamentos, ["Departamento", "Número de registros"])

In [ ]:
departamentos_1 = frecuencia_departamentos.iloc[0:24]
barras_h(departamentos_1, "Nombre del Departamento", "n", "Registros por departamento", "Número de registros", "Departamento",
         subtitulo="Nombres tal como vienen en el archivo, 1 a 24", alto=9, tam=7, total=len(df))

In [ ]:
departamentos_2 = frecuencia_departamentos.iloc[24:49]
barras_h(departamentos_2, "Nombre del Departamento", "n", "Registros por departamento", "Número de registros", "Departamento",
         subtitulo="Nombres tal como vienen en el archivo, 25 a 49", alto=9, tam=7, total=len(df))

```{admonition} Precaución
:class: warning
En la tabla el mismo departamento aparece escrito de varias formas, por ejemplo BOGOTÁ, D.C., BOGOTA, D.C. y Bogotá, D.C., o ANTIOQUIA y Antioquia. Por eso salen 49 departamentos. Abajo se normalizan los nombres (sin tildes y en mayúsculas).
```

In [ ]:
import unicodedata
def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t)).encode("ascii", "ignore").decode()
    return " ".join(t.upper().split())

df["Departamento_norm"] = df["Nombre del Departamento"].map(normalizar)
df["Municipio_norm"] = df["Nombre del Municipio"].map(normalizar)
tabla(pd.DataFrame({"Columna": ["Departamento", "Municipio"],
                    "Nombres en el archivo": [df["Nombre del Departamento"].nunique(), df["Nombre del Municipio"].nunique()],
                    "Nombres normalizados": [df["Departamento_norm"].nunique(), df["Municipio_norm"].nunique()]}))

In [ ]:
deptos_norm = df["Departamento_norm"].value_counts().rename_axis("Departamento").reset_index(name="n")
barras_h(deptos_norm, "Departamento", "n", "Registros por departamento", "Número de registros", "Departamento",
         subtitulo="Nombres normalizados", alto=8.5, tam=7, total=len(df))

```{admonition} Interpretación
:class: tip
Al normalizar quedan 24 departamentos y 193 municipios. Antioquia y Bogotá tienen la mitad de los registros, y después vienen Valle del Cauca y Cundinamarca. La Amazonía, la Orinoquía y casi todo el Pacífico no tienen monitoreo, así que los resultados nacionales representan sobre todo a las ciudades andinas.
```

#### Distribución por municipio

In [ ]:
municipios = (df.groupby("Municipio_norm")
                .agg(Departamento=("Departamento_norm", "first"), Registros=("Variable", "size"), Estaciones=("Estación", "nunique"),
                     Latitud=("Latitud", "mean"), Longitud=("Longitud", "mean"))
                .reset_index().sort_values("Registros", ascending=False))
mapa_satelital(municipios["Latitud"], municipios["Longitud"], (municipios["Registros"] ** 0.5) * 0.9 + 5,
               municipios["Municipio_norm"] + ": " + municipios["Registros"].astype(str) + " registros, " +
               municipios["Estaciones"].astype(str) + " estaciones",
               "Registros por municipio (el tamaño del punto es proporcional al número de registros)")

```{admonition} Interpretación
:class: tip
El mapa muestra lo mismo que la tabla de departamentos pero a nivel de municipio. Bogotá es el punto más grande, con 3.397 registros, seguido por Medellín (1.487) y Cali (1.036). Alrededor de Medellín aparecen varios puntos medianos (Itagüí, Girardota, Bello, Sabaneta), porque el Valle de Aburrá tiene una red densa.
```

In [ ]:
barras_h(municipios.head(20), "Municipio_norm", "Registros", "20 municipios con más registros", "Número de registros",
         "Municipio", subtitulo="Nombres normalizados", alto=8, total=len(df))

```{admonition} Interpretación
:class: tip
Los 20 municipios con más registros son capitales con redes grandes (Bogotá, Medellín, Cali, Bucaramanga, Santa Marta) y municipios industriales o mineros como Sogamoso y Nobsa (cemento y acero) o La Jagua de Ibirico (carbón), donde se vigilan fuentes de emisión puntuales.
```

### 1.6 Análisis específico de contaminantes

Para cada contaminante se muestra la estadística descriptiva y la distribución por región. Las regiones se definen por la latitud de la estación: sur (menos de 4°), centro (4° a 6°, Bogotá y Cundinamarca), centro-norte (6° a 8°, Antioquia, Santander y Boyacá) y Caribe (más de 8°).

In [ ]:
def estadisticos(serie, con_media=True):
    nombres = ["Mínimo", "Primer cuartil", "Mediana", "Media", "Tercer cuartil", "Máximo"]
    valores = [serie.min(), serie.quantile(0.25), serie.median(), serie.mean(), serie.quantile(0.75), serie.max()]
    if not con_media: nombres, valores = ["Mínimo", "Primer cuartil", "Mediana", "Máximo"], [serie.min(), serie.quantile(0.25), serie.median(), serie.max()]
    return pd.DataFrame({"Estadistico": nombres, "Valor": valores})

def promedio_anual(d, nombre): return d.groupby("Año", as_index=False).agg(**{nombre: ("Promedio", "mean"), "Registros": ("Promedio", "size")}).sort_values("Año").reset_index(drop=True)

def grafico_anual(d, col, titulo, subtitulo, ylab, anios=None):
    fig, ax = plt.subplots()
    ax.plot(d["Año"], d[col], linewidth=1.2, marker="o", markersize=6, color="black")
    for x, y in zip(d["Año"], d[col]): ax.text(x, y + (d[col].max() - d[col].min() + 1) * 0.03, f"{y:.1f}", ha="center", fontsize=9)
    ax.set_xticks(list(anios) if anios is not None else sorted(d["Año"].unique()))
    ax.set_ylim(top=d[col].max() * 1.08)
    ax.set_title(titulo + "\n" + subtitulo, fontweight="bold"); ax.set_xlabel("Año"); ax.set_ylabel(ylab)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout(); plt.show()

def cambio_porcentual(d, col, a1=2011, a2=2024):
    s = d.set_index("Año")[col]
    if a1 not in s.index or a2 not in s.index: return pd.DataFrame({"Periodo": ["N/A"], "Promedio_2011": [np.nan], "Promedio_2024": [np.nan], "Cambio_porcentual": [np.nan]})
    v1, v2 = s.loc[a1], s.loc[a2]
    return pd.DataFrame({"Periodo": [f"{a1}-{a2}"], "Promedio_2011": [v1], "Promedio_2024": [v2], "Cambio_porcentual": [(v2 - v1) / v1 * 100]})

# Distribución por región (según la latitud de la estación)
REGIONES = ["Sur (< 4°)", "Centro (4–6°)", "Centro-norte (6–8°)", "Caribe (> 8°)"]
df["Región"] = pd.cut(df["Latitud"], [-5, 4, 6, 8, 15], labels=REGIONES)

def resumen_region(d):
    return (d.groupby("Región", observed=True)["Promedio"]
              .agg(Registros="size", Mediana="median", Q1=lambda s: s.quantile(.25), Q3=lambda s: s.quantile(.75),
                   Maximo="max").reset_index())

def grafico_region(d, nombre, unidad, norma=None):
    grupos = [g for g in REGIONES if (d["Región"] == g).any()]
    datos = [d.loc[d["Región"] == g, "Promedio"].dropna() for g in grupos]
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.boxplot(datos, widths=0.5)
    ax.set_xticks(range(1, len(grupos) + 1), [f"{g}\nn = {len(x)}" for g, x in zip(grupos, datos)])
    if norma:
        ax.axhline(norma[0], ls="--", color="black", lw=1, label=f"Norma anual ({norma[0]})")
        ax.axhline(norma[1], ls=":", color="gray", lw=1.2, label=f"Meta 2030 ({norma[1]})")
        ax.legend(loc="upper right")
    ax.set_title(f"Distribución de {nombre} por región", fontweight="bold")
    ax.set_ylabel(f"Promedio ({unidad})"); ax.grid(axis="x", visible=False)
    plt.tight_layout(); plt.show()

#### PM10

In [ ]:
pm10 = df[df["Variable"] == "PM10"].copy()
print("Registros de PM10:", len(pm10))
estadistica_pm10 = estadisticos(pm10["Promedio"])
tabla(estadistica_pm10, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
El PM10 tiene 2.984 registros. La media (37,1 µg/m³) es algo mayor que la mediana (34,6 µg/m³), así que la distribución tiene cola hacia la derecha. El máximo es 162,3 µg/m³.
```

In [ ]:
pm10_anual = promedio_anual(pm10, "Promedio_PM10")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(pm10), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(pm10, "PM10", "µg/m³", norma=(50, 30))

```{admonition} Interpretación
:class: tip
Las medianas de PM10 son parecidas entre regiones, entre 32,2 µg/m³ en el centro (Bogotá y alrededores) y 37,4 µg/m³ en el centro-norte (Valle de Aburrá). En todas las regiones la mediana está por debajo de la norma anual de 50 µg/m³, pero por encima de la meta de 2030 (30 µg/m³).
```

In [ ]:
tabla_cambio_pm10 = cambio_porcentual(pm10_anual, "Promedio_PM10")
tabla(tabla_cambio_pm10, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
Entre 2011 y 2024 el promedio nacional de PM10 bajó 27,75 %.
```

#### PM2.5

In [ ]:
pm25 = df[df["Variable"] == "PM2.5"].copy()
print("Registros de PM2.5:", len(pm25))
estadistica_pm25 = estadisticos(pm25["Promedio"])
tabla(estadistica_pm25, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
El PM2.5 tiene 1.984 registros y valores más bajos que el PM10, porque es la fracción fina del material particulado. La media (16,8 µg/m³) y la mediana (15,7 µg/m³) son cercanas.
```

In [ ]:
pm25_anual = promedio_anual(pm25, "Promedio_PM25")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(pm25), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(pm25, "PM2.5", "µg/m³", norma=(25, 15))

```{admonition} Interpretación
:class: tip
El centro-norte tiene la mediana más alta de PM2.5 (17,8 µg/m³). Ahí está el Valle de Aburrá, donde las montañas encierran los contaminantes. El Caribe tiene la más baja (13,7 µg/m³). Todas las medianas están por debajo de la norma anual de 25 µg/m³, y la del centro-norte es la única que pasa la meta de 2030 (15 µg/m³).
```

In [ ]:
tabla_cambio_pm25 = cambio_porcentual(pm25_anual, "Promedio_PM25")
tabla(tabla_cambio_pm25, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
Entre 2011 y 2024 el promedio nacional de PM2.5 bajó 31,35 %.
```

#### NO₂

In [ ]:
no2 = df[df["Variable"] == "NO2"].copy()
print("Registros de NO2:", len(no2))
estadistica_no2 = estadisticos(no2["Promedio"])
tabla(estadistica_no2, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
El NO₂ tiene 644 registros. La media (26,5 µg/m³) y la mediana (26,0 µg/m³) son casi iguales.
```

In [ ]:
no2_anual = promedio_anual(no2, "Promedio_NO2")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(no2), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(no2, "NO₂", "µg/m³", norma=(60, 40))

```{admonition} Interpretación
:class: tip
El NO₂ es el contaminante con más diferencia entre regiones. En el centro y el centro-norte la mediana está entre 26,4 y 30,0 µg/m³, más del doble que en el sur (13,4) y el Caribe (10,8). El NO₂ viene sobre todo del tráfico vehicular, que es mayor en las ciudades andinas. El sur y el Caribe tienen pocos registros (29 y 25). Todos los valores están lejos de la norma anual de 60 µg/m³.
```

In [ ]:
tabla_cambio_no2 = cambio_porcentual(no2_anual, "Promedio_NO2")
tabla(tabla_cambio_no2, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
Entre 2011 y 2024 el promedio nacional de NO₂ bajó 25,9 %.
```

#### O₃

In [ ]:
o3 = df[df["Variable"] == "O3"].copy()
print("Registros de O3:", len(o3))
estadistica_o3 = estadisticos(o3["Promedio"])
tabla(estadistica_o3, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
El ozono tiene 827 registros, con media y mediana cercanas (26,2 y 25,7 µg/m³).
```

In [ ]:
o3_anual = promedio_anual(o3, "Promedio_O3")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(o3), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(o3, "O₃", "µg/m³")

```{admonition} Interpretación
:class: tip
El ozono es más alto en el Caribe (mediana de 30,5 µg/m³) y más bajo en el centro (22,4 µg/m³). El ozono se forma con la luz solar, y el Caribe es la región con más radiación (ver RGlobal más abajo).
```

In [ ]:
tabla_cambio_o3 = cambio_porcentual(o3_anual, "Promedio_O3")
tabla(tabla_cambio_o3, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
El ozono es el único contaminante cuyo promedio nacional subió entre 2011 y 2024 (14 %).
```

#### SO₂

In [ ]:
so2 = df[df["Variable"] == "SO2"].copy()
print("Registros de SO2:", len(so2))
estadistica_so2 = estadisticos(so2["Promedio"])
tabla(estadistica_so2, ["Estadístico", "Valor"])

```{admonition} Interpretación
:class: tip
El SO₂ tiene cola larga hacia la derecha: la media (9,4 µg/m³) es mucho mayor que la mediana (6,0 µg/m³) y el máximo llega a 91,7 µg/m³.
```

In [ ]:
so2_anual = promedio_anual(so2, "Promedio_SO2")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(so2), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(so2, "SO₂", "µg/m³")

```{admonition} Interpretación
:class: tip
El SO₂ es más alto en el centro-norte (mediana de 8,8 µg/m³) y muy bajo en el Caribe (2,9 µg/m³). Los valores altos aparecen como puntos sueltos sobre las cajas y corresponden a pocas estaciones cerca de industrias.
```

In [ ]:
tabla_cambio_so2 = cambio_porcentual(so2_anual, "Promedio_SO2")
tabla(tabla_cambio_so2, ["Periodo", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

```{admonition} Interpretación
:class: tip
Entre 2011 y 2024 el promedio nacional de SO₂ bajó 34,8 %, pero ese promedio depende de pocas estaciones industriales que entran y salen de la red.
```

#### Comparación del cambio porcentual de los contaminantes

In [ ]:
cambios = []
for nombre, dat in [("PM10", pm10_anual), ("PM2.5", pm25_anual), ("NO2", no2_anual), ("O3", o3_anual), ("SO2", so2_anual)]:
    s = dat.set_index("Año").iloc[:, 0]
    if 2011 in s.index and 2024 in s.index:
        v1, v2 = s.loc[2011], s.loc[2024]
        cambios.append({"Contaminante": nombre, "Promedio_2011": v1, "Promedio_2024": v2, "Cambio_porcentual": (v2 - v1) / v1 * 100})
comparacion_contaminantes = pd.DataFrame(cambios)
tabla(comparacion_contaminantes, ["Contaminante", "Promedio 2011", "Promedio 2024", "Cambio (%)"])

In [ ]:
d = comparacion_contaminantes.sort_values("Cambio_porcentual")
fig, ax = plt.subplots()
ax.barh(d["Contaminante"], d["Cambio_porcentual"], color="#595959")
for y, v in zip(d["Contaminante"], d["Cambio_porcentual"]): ax.text(v * 1.03, y, f"{v:.1f}%", va="center", ha="right" if v < 0 else "left", fontsize=9)
ax.set_xlim(d["Cambio_porcentual"].min() * 1.15, d["Cambio_porcentual"].max() * 1.15)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Cambio porcentual de los principales contaminantes\nVariación del promedio entre 2011 y 2024", fontweight="bold")
ax.set_xlabel("Cambio porcentual (%)"); ax.set_ylabel("Contaminante"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
Cuatro de los cinco contaminantes bajaron entre 2011 y 2024. El material particulado bajó cerca de 30 % y el ozono subió 14 %.
```

### 1.7 Análisis de variables meteorológicas

#### Temperatura del aire (TAire)

In [ ]:
taire = df[df["Variable"] == "TAire"].copy()
print("Registros de TAire:", len(taire))
estadistica_taire = estadisticos(taire["Promedio"])
tabla(estadistica_taire, ["Estadístico", "Valor"])

In [ ]:
taire_anual = promedio_anual(taire, "Promedio_TAire")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(taire), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(taire, "TAire", "°C")

```{admonition} Interpretación
:class: tip
TAire solo tiene registros en el centro (22) y el centro-norte (51). Con 73 registros no se puede comparar entre regiones. TAire2 y TAire10 tienen 834 y 334 registros y sirven mejor para estudiar la temperatura.
```

#### Humedad relativa del aire (HAire)

In [ ]:
haire = df[df["Variable"] == "HAire"].copy()
print("Registros de HAire:", len(haire))
estadistica_haire = estadisticos(haire["Promedio"])
tabla(estadistica_haire, ["Estadístico", "Valor"])

In [ ]:
haire_anual = promedio_anual(haire, "Promedio_HAire")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(haire), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(haire, "HAire", "%")

```{admonition} Interpretación
:class: tip
La humedad relativa tiene medianas entre 66,9 % (sur) y 73,6 % (centro-norte). No hay registros en el Caribe y el sur solo tiene 6.
```

#### Velocidad del viento (VViento)

In [ ]:
vviento = df[df["Variable"] == "VViento"].copy()
print("Registros de VViento:", len(vviento))
estadistica_vviento = estadisticos(vviento["Promedio"])
tabla(estadistica_vviento, ["Estadístico", "Valor"])

In [ ]:
vviento_anual = promedio_anual(vviento, "Promedio_VViento")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(vviento), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(vviento, "VViento", "m/s")

```{admonition} Interpretación
:class: tip
El viento es débil en todas las regiones, con medianas entre 1,1 y 1,4 m/s. El centro-norte tiene el valor más bajo (1,1 m/s) y también la mediana más alta de PM2.5. Con poco viento los contaminantes se dispersan menos.
```

#### Presión atmosférica (P)

In [ ]:
presion = df[df["Variable"] == "P"].copy()
print("Registros de P:", len(presion))
estadistica_presion = estadisticos(presion["Promedio"])
tabla(estadistica_presion, ["Estadístico", "Valor"])

In [ ]:
presion_anual = promedio_anual(presion, "Promedio_P")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(presion), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(presion, "presión (P)", "mmHg")

```{admonition} Interpretación
:class: tip
Las medianas de presión siguen la altitud: 565,6 mmHg en el centro (Bogotá está a unos 2.600 m) y 755,6 mmHg en el Caribe, casi a nivel del mar. Eso indica que la mayoría de los datos son correctos. Los puntos por encima de 800 mmHg no corresponden a ninguna altitud posible. La celda siguiente muestra de dónde salen.
```

In [ ]:
sospechosos = presion[presion["Promedio"] > 800]
print("Registros de presión por encima de 800 mmHg:", len(sospechosos), "de", len(presion))
tabla(sospechosos["Nombre del Municipio"].value_counts().head(6).rename_axis("Municipio").reset_index(name="Registros"))

```{admonition} Precaución
:class: warning
Hay registros de más de 800 mmHg en Cali (22), Villavicencio (18) y Barranquilla (18). En Cali, a unos 1.000 m de altura, la presión real es de unos 680 mmHg, y ni siquiera a nivel del mar pasa de 760 mmHg. Los valores cercanos a 1.010 son hectopascales (hPa) marcados como mmHg. Esta variable no se puede usar sin corregir las unidades.
```

#### Precipitación líquida (PLiquida)

In [ ]:
pliquida = df[df["Variable"] == "PLiquida"].copy()
print("Registros de PLiquida:", len(pliquida))
estadistica_pliquida = estadisticos(pliquida["Promedio"])
tabla(estadistica_pliquida, ["Estadístico", "Valor"])

In [ ]:
pliquida_anual = promedio_anual(pliquida, "Promedio_PLiquida")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(pliquida), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(pliquida, "PLiquida", "mm")

```{admonition} Interpretación
:class: tip
Las medianas de precipitación son bajas en todas las regiones, entre 1,2 y 2,3 mm, pero hay valores muy lejos de las cajas. Parecen acumulados reportados como promedio, por eso en esta variable se usa la mediana.
```

#### Radiación global (RGlobal)

In [ ]:
rglobal = df[df["Variable"] == "RGlobal"].copy()
print("Registros de RGlobal:", len(rglobal))
estadistica_rglobal = estadisticos(rglobal["Promedio"])
tabla(estadistica_rglobal, ["Estadístico", "Valor"])

In [ ]:
rglobal_anual = promedio_anual(rglobal, "Promedio_RGlobal")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(rglobal), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(rglobal, "RGlobal", "W/m²")

```{admonition} Interpretación
:class: tip
La radiación global es mayor en el Caribe (mediana de 219,1 W/m²) y menor en el centro (168,3 W/m²), donde hay más nubosidad.
```

#### Radiación ultravioleta B (RUVb)

In [ ]:
ruvb = df[df["Variable"] == "RUVb"].copy()
print("Registros de RUVb:", len(ruvb))
estadistica_ruvb = estadisticos(ruvb["Promedio"])
tabla(estadistica_ruvb, ["Estadístico", "Valor"])

In [ ]:
ruvb_anual = promedio_anual(ruvb, "Promedio_RUVb")   # se usa para el cambio entre 2011 y 2024
tabla(resumen_region(ruvb), ["Región", "Registros", "Mediana", "Q1", "Q3", "Máximo"])

In [ ]:
grafico_region(ruvb, "RUVb", "MED/h")

```{admonition} Interpretación
:class: tip
La RUVb tiene medianas muy distintas entre regiones, 1,9 en el centro-norte y 194,9 en el centro, que no tienen explicación física. La diferencia viene de las autoridades que reportan y de la escala que usa cada una, como se ve en la celda siguiente.
```

In [ ]:
autoridades_ruvb = (ruvb.groupby("Año")["Autoridad Ambiental"].unique().apply(lambda a: ", ".join(sorted(a)))
                    .reset_index(name="Autoridades que reportan"))
tabla(autoridades_ruvb)

```{admonition} Precaución
:class: warning
Hasta 2014 la RUVb solo la reportaba la SDA (Bogotá). Desde 2017 la reportan otras autoridades (CDMB, Corpoboyacá, CRC) con otra escala. Los dos periodos no se pueden comparar y cualquier correlación con RUVb sale afectada por este cambio.
```

### 1.8 Análisis de correlaciones

#### Preparación de los datos

In [ ]:
datos_correlacion = (df.groupby(["Año", "Variable"], as_index=False)["Promedio"].mean()
                     .pivot(index="Año", columns="Variable", values="Promedio").reset_index().rename_axis(None, axis=1))
tabla(datos_correlacion[["Año", "PM10", "PM2.5", "NO2", "O3", "SO2", "TAire", "VViento", "P"]].head())

```{admonition} Interpretación
:class: tip
Para comparar variables que se miden en estaciones distintas se usa el promedio nacional de cada año. Así cada variable queda con 14 puntos como máximo, y algunas con menos (TAire 6, HAire 7). Por eso se usa Spearman, que trabaja con rangos y es menos sensible a un año con un valor extremo que Pearson.
```

#### Matriz de correlaciones

In [ ]:
variables_eda = ["PM10", "PM2.5", "NO2", "SO2", "O3", "TAire", "HAire", "VViento", "P", "PLiquida", "RGlobal", "RUVb"]
matriz_correlacion = datos_correlacion[[v for v in variables_eda if v in datos_correlacion.columns]].corr(method="spearman")

In [ ]:
distancia = 1 - matriz_correlacion
orden = leaves_list(linkage(squareform(distancia.values, checks=False), method="complete"))
m = matriz_correlacion.iloc[orden, orden]
mascara = np.tril(np.ones(m.shape, dtype=bool))
fig, ax = plt.subplots(figsize=(9, 7.5))
sns.heatmap(m, mask=mascara, annot=True, fmt=".2f", annot_kws={"size": 8}, cmap="RdBu_r", vmin=-1, vmax=1, square=True, linewidths=0.5, cbar_kws={"shrink": 0.7}, ax=ax)
ax.set_title("Matriz de correlación de Spearman", fontweight="bold")
plt.xticks(rotation=45, ha="left"); ax.xaxis.tick_top()
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
Con Spearman la matriz forma dos grupos. Uno con PM10, PM2.5, NO₂, O₃, viento y humedad, que se relacionan de forma positiva. Otro con radiación global, presión, temperatura y RUVb, que en general tiene relación negativa con los contaminantes.

- PM10 y PM2.5: ρ = 0,96 con 14 años. Es la relación más confiable. El PM2.5 es parte del PM10 y los dos vienen de las mismas fuentes.
- PM2.5 y O₃: ρ = 0,64 con 14 años, relación fuerte con la serie completa.
- TAire con PM10 (ρ = −0,94) y con NO₂ (ρ = −0,83): tienen sentido físico porque con más temperatura el aire se mezcla mejor, pero TAire solo tiene 6 años.
- RUVb con PM10 (−0,93) y con TAire (0,94), y NO₂ con P (−0,77): no se pueden usar por los problemas de RUVb y de la presión.

Una correlación alta no indica que una variable cause la otra.
```

#### Tabla de correlaciones y fuerza de asociación

In [ ]:
def fuerza(r):
    a = abs(r)
    if a >= 0.80: return "Muy fuerte"
    elif a >= 0.60: return "Fuerte"
    elif a >= 0.40: return "Moderada"
    elif a >= 0.20: return "Débil"
    else: return "Muy débil"

completos = datos_correlacion.select_dtypes("number").dropna()
m = completos.corr(method="spearman")
pares = [(a, b, m.loc[a, b]) for k, a in enumerate(m.columns) for b in m.columns[k + 1:]]
correlaciones_todas = pd.DataFrame(pares, columns=["Variable 1", "Variable 2", "Correlacion"])
correlaciones_todas["Fuerza"] = correlaciones_todas["Correlacion"].apply(fuerza)
orden_abs = correlaciones_todas["Correlacion"].abs().sort_values(ascending=False).index
correlaciones_todas = correlaciones_todas.loc[orden_abs].reset_index(drop=True)
print("Años con dato en todas las variables:", ", ".join(completos["Año"].astype(str)))
print("Pares con |ρ| ≥ 0,60:", (correlaciones_todas["Correlacion"].abs() >= 0.6).sum(), "de", len(correlaciones_todas))

In [ ]:
def color_corr(x):
    if x >= 0.80: return "#81C784"
    elif x >= 0.60: return "#A5D6A7"
    elif x <= -0.80: return "#E57373"
    elif x <= -0.60: return "#EF9A9A"
    else: return "#E0E0E0"

principales = ["Año"] + variables_eda
fuertes = correlaciones_todas[(correlaciones_todas["Correlacion"].abs() >= 0.6)
                              & correlaciones_todas["Variable 1"].isin(principales)
                              & correlaciones_todas["Variable 2"].isin(principales)]
tabla(fuertes, ["Variable 1", "Variable 2", "ρ de Spearman", "Fuerza"]).map(
    lambda v: "background-color: " + color_corr(v) + "; color: black", subset=["ρ de Spearman"])

```{admonition} Interpretación
:class: tip
Esta tabla usa .dropna() sobre todas las variables, así que solo quedan los años con dato en todas: 2011, 2012, 2013, 2017, 2018 y 2019. Se muestran los pares con |ρ| de 0,60 o más entre las 12 variables principales y el año. Con solo 6 años los valores cambian frente a la matriz. Por ejemplo, PM10 y PM2.5 da 0,89 aquí y 0,96 en la matriz.

Las filas con Año muestran la tendencia: PM10 (−0,94), NO₂ (−0,83) y PM2.5 (−0,77) bajan con el tiempo, y P (0,77) y RGlobal (0,83) suben. En la presión es otra vez el problema de unidades.
```

### 1.9 Análisis de valores atípicos

In [ ]:
outliers_pm = pd.DataFrame({"Variable": ["PM10", "PM2.5"], "Q1": [pm10["Promedio"].quantile(0.25), pm25["Promedio"].quantile(0.25)], "Q3": [pm10["Promedio"].quantile(0.75), pm25["Promedio"].quantile(0.75)]})
outliers_pm["Rango_IQR"] = outliers_pm["Q3"] - outliers_pm["Q1"]
outliers_pm["Limite_inferior"] = outliers_pm["Q1"] - 1.5 * outliers_pm["Rango_IQR"]
outliers_pm["Limite_superior"] = outliers_pm["Q3"] + 1.5 * outliers_pm["Rango_IQR"]
tabla(outliers_pm, ["Variable", "Primer cuartil", "Tercer cuartil", "Rango IQR", "Límite inferior", "Límite superior"])

```{admonition} Interpretación
:class: tip
Para PM10 los valores atípicos son los que pasan de 71,5 µg/m³ y para PM2.5 los que pasan de 30,75 µg/m³. Los límites inferiores salen negativos o casi cero, así que no hay atípicos bajos.
```

#### Valores atípicos de PM10

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.boxplot(pm10["Promedio"].dropna(), widths=0.4)
ax.set_title("Distribución y valores atípicos de PM10", fontweight="bold"); ax.set_ylabel("Concentración de PM10"); ax.set_xticks([])
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
El PM10 tiene 91 atípicos (3,0 %), todos por encima de la caja. El más alto es la estación Bolivia de Bogotá, con 162 µg/m³ en 2012. También aparecen zonas industriales como Yumbo y Ráquira.
```

#### Valores atípicos de PM2.5

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.boxplot(pm25["Promedio"].dropna(), widths=0.4)
ax.set_title("Distribución y valores atípicos de PM2.5", fontweight="bold"); ax.set_ylabel("Concentración de PM2.5"); ax.set_xticks([])
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
El PM2.5 tiene 74 atípicos (3,7 %), también solo valores altos, y se concentran en pocas estaciones.
```

#### Valores atípicos de variables meteorológicas

In [ ]:
meteorologicas = df[df["Variable"].isin(["TAire", "HAire", "VViento", "P", "PLiquida", "RGlobal", "RUVb"])]
outliers_meteorologicas = meteorologicas.groupby("Variable", as_index=False).agg(Q1=("Promedio", lambda s: s.quantile(0.25)), Q3=("Promedio", lambda s: s.quantile(0.75)))
outliers_meteorologicas["IQR"] = outliers_meteorologicas["Q3"] - outliers_meteorologicas["Q1"]
outliers_meteorologicas["Limite_inferior"] = outliers_meteorologicas["Q1"] - 1.5 * outliers_meteorologicas["IQR"]
outliers_meteorologicas["Limite_superior"] = outliers_meteorologicas["Q3"] + 1.5 * outliers_meteorologicas["IQR"]
tabla(outliers_meteorologicas, ["Variable", "Primer cuartil", "Tercer cuartil", "Rango IQR", "Límite inferior", "Límite superior"])

In [ ]:
orden_met = sorted(meteorologicas["Variable"].unique())
fig, ax = plt.subplots()
ax.boxplot([meteorologicas.loc[meteorologicas["Variable"] == v, "Promedio"].dropna() for v in orden_met])
ax.set_xticks(range(1, len(orden_met) + 1))
ax.set_xticklabels(orden_met)
ax.set_title("Distribución y valores atípicos de las variables meteorológicas", fontweight="bold"); ax.set_xlabel("Variable meteorológica"); ax.set_ylabel("Valor registrado")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
La presión tiene atípicos altos, que son los registros en hPa. La precipitación tiene valores muy lejos de la caja y el viento tiene pocos atípicos.

Los atípicos no se eliminan porque pueden ser episodios reales de contaminación. Solo se deben corregir los errores que se pueden demostrar, como la presión en hPa o la representatividad mayor a 100 %.
```

### 1.10 Resumen estadístico general

In [ ]:
variables_resumen = df[df["Variable"].isin(["PM10", "PM2.5", "SO2", "NO2", "O3", "TAire", "HAire", "VViento", "P", "PLiquida", "RGlobal", "RUVb"])].groupby("Variable", as_index=False).agg(N=("Promedio", "count"), Mínimo=("Promedio", "min"), Q1=("Promedio", lambda s: s.quantile(0.25)), Mediana=("Promedio", "median"), Media=("Promedio", "mean"), Q3=("Promedio", lambda s: s.quantile(0.75)), Máximo=("Promedio", "max"))
tabla(variables_resumen, ["Variable", "N", "Mínimo", "Q1", "Mediana", "Media", "Q3", "Máximo"])

```{admonition} Interpretación
:class: tip
La tabla junta en un solo lugar las estadísticas de las 12 variables principales.
```

### 1.11 Variables con problemas

Estas son las variables o columnas que no se pueden usar tal como vienen en el archivo.

| Variable | Problema | Causa probable | Decisión |
|---|---|---|---|
| P (presión) | 108 registros por encima de 800 mmHg, sobre todo en Cali, Villavicencio y Barranquilla | Valores en hPa etiquetados como mmHg | No usar sin convertir unidades |
| RUVb | Pasa de 0,6 (2011 a 2014) a valores de 130 a 180 desde 2019 | Cambio de autoridad que reporta y de escala | No es comparable, no se usa en correlaciones |
| TAire | 73 registros en 6 años y 24 % de representatividad | Pocas estaciones reportan sin altura | Usar TAire2 o TAire10 |
| HAire | 75 registros en 7 años y 27 % de representatividad | Pocas estaciones reportan sin altura | Usar HAire2 o HAire10 |
| DViento | El promedio aritmético de grados no tiene sentido físico | Variable circular | Requiere estadística circular |
| PLiquida | Media de 9,8 mm frente a mediana de 1,4 mm, máximo de 3.236,7 mm | Acumulados reportados como promedios | Usar la mediana |
| Representatividad Temporal | 12 registros por encima de 100 % (máximo 167 %) | Error de captura | Se documenta, no se corrige |
| Excedencias en registros de 1 h | Máximos de 3.608 µg/m³ de PM2.5 sin excedencias | No hay límite horario de PM2.5 en la norma | Usar solo registros de 24 h |
| Departamento y municipio | 49 nombres para 24 departamentos y 339 para 193 municipios | Mayúsculas, minúsculas y tildes | Normalizar los nombres |


### 1.12 Variable objetivo: excedencia diaria de PM2.5

El modelo va a responder si una estación tendrá en el año al menos un día con PM2.5 por encima del límite diario de la Resolución 2254 de 2017. Primero hay que revisar qué mide la columna Excedencias limite actual en cada tiempo de exposición.

In [ ]:
pm25["Excede"] = pm25["Excedencias limite actual"] > 0
verificacion = pm25.groupby("Tiempo de exposición (horas)").agg(
    Registros=("Excede", "size"),
    Porcentaje_que_excede=("Excede", lambda s: s.mean() * 100),
    Maximo_sin_excedencia=("Máximo", lambda s: s[~pm25.loc[s.index, "Excede"]].max()),
).reset_index()
tabla(verificacion, ["Exposición (h)", "Registros", "% que excede", "Máximo sin excedencia (µg/m³)"], decimales=1)

```{admonition} Interpretación
:class: tip
En los registros de 24 horas ninguna estación sin excedencias pasa de 50,2 µg/m³, que coincide con el límite diario. En los de 1 hora hay estaciones con máximos de 3.608 µg/m³ que aparecen sin excedencias, porque la norma no tiene límite horario para PM2.5. Por eso la variable objetivo se arma solo con los registros de 24 horas.
```

In [ ]:
pm25_24h = pm25[pm25["Tiempo de exposición (horas)"] == 24].copy()
pm25_24h["Excede"] = pm25_24h["Excede"].astype(int)
print("Registros:", len(pm25_24h), "| Estaciones:", pm25_24h["Estación"].nunique())
balance = pm25_24h["Excede"].value_counts().rename_axis("Excede").reset_index(name="Registros")
balance["Excede"] = balance["Excede"].map({0: "No", 1: "Sí"})
balance["Porcentaje"] = balance["Registros"] / balance["Registros"].sum() * 100
tabla(balance, ["Excede", "Registros", "Porcentaje (%)"], decimales=1)

```{admonition} Interpretación
:class: tip
La variable objetivo tiene 1.144 registros de 383 estaciones y el 47,9 % excede el límite. Las dos clases tienen casi el mismo tamaño, así que no hace falta sobremuestrear y se puede usar la exactitud junto con el AUC y el recall.
```

In [ ]:
fig, ax = plt.subplots()
for valor, color, etiqueta in [(0, "#9e9e9e", "No excede"), (1, "#c62828", "Excede")]:
    d = pm25_24h[pm25_24h["Excede"] == valor]
    ax.scatter(d["Año"] + np.random.uniform(-0.25, 0.25, len(d)), d["Máximo"], s=12, alpha=0.6, color=color, label=etiqueta)
ax.step([2010.5, 2018.5, 2024.5], [50, 37, 37], where="post", color="black", linewidth=1.5, label="Límite diario (Res. 2254/2017)")
ax.set_ylim(0, 150); ax.set_xticks(range(2011, 2025))
ax.set_title("Máximo diario de PM2.5 y límite de la norma", fontweight="bold")
ax.set_xlabel("Año"); ax.set_ylabel("Máximo diario de PM2.5 (µg/m³)")
plt.xticks(rotation=45, ha="right"); ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
Los registros que exceden (rojo) quedan por encima de la línea del límite y los que no (gris) por debajo. El límite bajó de 50 a 37 µg/m³ a mediados de 2018, así que una misma concentración puede ser excedencia después de 2018 y no antes.
```

In [ ]:
def barras_tasa(d, cat, titulo, xlab, subtitulo=None, alto=6, horizontal=True):
    """Barras con la tasa de excedencia. Etiqueta: número de registros arriba y (%) que excede debajo."""
    fig, ax = plt.subplots(figsize=(9, alto))
    etiquetas = [f"{n}\n({t:.0f} %)" for n, t in zip(d["n"], d["Tasa"])]
    if horizontal:
        d = d.sort_values("Tasa")
        etiquetas = [f"{n}\n({t:.0f} %)" for n, t in zip(d["n"], d["Tasa"])]
        ax.barh(d[cat].astype(str), d["Tasa"], color="#4a6572")
        for y, t, e in zip(d[cat].astype(str), d["Tasa"], etiquetas):
            ax.text(t + 1, y, e, va="center", fontsize=8, linespacing=0.95)
        ax.set_xlim(0, 100); ax.set_xlabel(xlab); ax.grid(axis="y", visible=False)
    else:
        ax.bar(d[cat].astype(str), d["Tasa"], color="#4a6572")
        for x, t, e in zip(d[cat].astype(str), d["Tasa"], etiquetas):
            ax.text(x, t + 1.5, e, ha="center", fontsize=8, linespacing=0.95)
        ax.set_ylim(0, 100); ax.set_ylabel(xlab); ax.grid(axis="x", visible=False)
        plt.xticks(rotation=45, ha="right")
    ax.set_title(titulo if subtitulo is None else titulo + "\n" + subtitulo, fontweight="bold")
    plt.tight_layout(); plt.show()

por_anio = pm25_24h.groupby("Año").agg(Tasa=("Excede", lambda s: s.mean() * 100), n=("Excede", "size")).reset_index()
barras_tasa(por_anio, "Año", "Porcentaje de registros que exceden el límite por año", "Registros que exceden (%)",
            subtitulo="Arriba: número de registros. Entre paréntesis: porcentaje que excede", horizontal=False)

```{admonition} Interpretación
:class: tip
El porcentaje de excedencia no tiene una tendencia clara. Va de 21 % en 2011 (14 registros) a 69 % en 2020. Entre 2011 y 2016 hay pocos registros por año y en 2018 el límite se volvió más estricto, lo que complica la comparación. Aunque el promedio de PM2.5 bajó, los días con excedencia siguen siendo frecuentes.
```

In [ ]:
por_tipo = pm25_24h.groupby("Tipo de Estación").agg(Tasa=("Excede", lambda s: s.mean() * 100), n=("Excede", "size")).reset_index()
tabla(por_tipo, ["Tipo de estación", "% que excede", "Registros"], decimales=1)

```{admonition} Interpretación
:class: tip
Las estaciones fijas exceden mucho más (52 %) que las indicativas (20 %). Las fijas miden todo el año y suelen estar en zonas urbanas, mientras que las indicativas son campañas cortas con menos días para registrar una excedencia.
```

In [ ]:
por_depto = pm25_24h.groupby("Departamento_norm").agg(Tasa=("Excede", lambda s: s.mean() * 100), n=("Excede", "size")).reset_index()
por_depto = por_depto[por_depto["n"] >= 20]
barras_tasa(por_depto, "Departamento_norm", "Porcentaje de registros que exceden el límite de PM2.5",
            "Registros que exceden (%)", subtitulo="Departamentos con al menos 20 registros", alto=6)

```{admonition} Interpretación
:class: tip
Bogotá (65 %), Antioquia (57 %) y Cundinamarca (55 %) exceden con frecuencia, mientras que Córdoba (5 %), Magdalena (15 %) y Bolívar (22 %) casi no. Las ciudades andinas están en valles o altiplanos donde hay inversiones térmicas y poco viento, y en la costa los vientos alisios ayudan a dispersar. La ubicación debería ser uno de los predictores más importantes.
```

In [ ]:
estaciones = pm25_24h.groupby("Estación").agg(Latitud=("Latitud", "mean"), Longitud=("Longitud", "mean"),
                                            Tasa=("Excede", "mean"), n=("Excede", "size")).reset_index()
estaciones["Tasa"] = estaciones["Tasa"] * 100
mapa_satelital(estaciones["Latitud"], estaciones["Longitud"], 7 + estaciones["n"] * 1.6,
               estaciones["Estación"] + ": excede en " + estaciones["Tasa"].round(0).astype(int).astype(str) +
               " % de " + estaciones["n"].astype(str) + " años",
               "Estaciones de PM2.5 según el porcentaje de años con excedencia", color=estaciones["Tasa"],
               escala=[[0, "#7fdcff"], [0.5, "#ffd23f"], [1, "#ff3b3b"]], titulo_barra="% de años")

```{admonition} Interpretación
:class: tip
Los puntos rojos se concentran en Bogotá y el Valle de Aburrá, y los azules en el Caribe y el suroccidente. El tamaño del punto indica cuántos años reportó la estación.
```

In [ ]:
pm25_24h["Cuartil_datos"] = pd.qcut(pm25_24h["No. de datos"], 4, labels=["Q1 (menos días)", "Q2", "Q3", "Q4 (más días)"])
esfuerzo = pm25_24h.groupby("Cuartil_datos", observed=True).agg(
    Dias_medidos=("No. de datos", "median"), Tasa=("Excede", lambda s: s.mean() * 100), n=("Excede", "size")).reset_index()
tabla(esfuerzo, ["Cuartil de días medidos", "Mediana de días medidos", "% que excede", "Registros"], decimales=1)

```{admonition} Interpretación
:class: tip
Las estaciones que midieron pocos días (Q1, mediana de 42 días) exceden en el 21 % de los casos y las que midieron casi todo el año (Q4) en el 79 %. Entre más días se mide, más fácil es captar al menos un día malo.
```

```{admonition} Precaución
:class: warning
La variable objetivo depende de cuántos días se midió y no solo de la calidad del aire. El modelo tiene que incluir No. de datos o la representatividad como variable de control. Si no, va a confundir una estación que mide mucho con una estación contaminada.
```

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.boxplot([pm25_24h.loc[pm25_24h["Excede"] == 0, "Promedio"], pm25_24h.loc[pm25_24h["Excede"] == 1, "Promedio"]],
           widths=0.5)
ax.set_xticks([1, 2], [f"No excede\nn = {(pm25_24h.Excede == 0).sum()}", f"Excede\nn = {(pm25_24h.Excede == 1).sum()}"])
ax.set_title("Promedio anual de PM2.5 según excedencia", fontweight="bold")
ax.set_ylabel("Promedio anual de PM2.5 (µg/m³)"); plt.tight_layout(); plt.show()

```{admonition} Interpretación
:class: tip
Las estaciones que exceden tienen un promedio anual más alto (mediana de 18,4 frente a 13,1 µg/m³), pero las cajas se cruzan. Hay estaciones con promedio bajo que igual tienen días críticos, así que el promedio solo no alcanza para saber si habrá excedencias.
```

```{admonition} Precaución
:class: warning
El promedio, la mediana, el percentil 98, el máximo, los días de excedencia y el porcentaje de excedencias del PM2.5 se calculan con las mismas mediciones que definen la excedencia. Si se usan como predictores el modelo hace trampa (fuga de información o data leakage).
```

#### Predictores candidatos para el modelo

| Variable | Uso | Razón |
|---|---|---|
| Latitud, longitud, departamento | Incluir | Diferencia clara entre la región andina y el Caribe |
| Tipo de estación | Incluir | Fijas 52 % e indicativas 20 % |
| No. de datos, representatividad | Incluir como control | Controlan cuántos días se midió |
| Año | Incluir | Recoge el cambio de límite en 2018 y los cambios de la red |
| Autoridad ambiental | Evaluar | Puede repetir la información de ubicación |
| Promedio, mediana, P98, máximo y días de excedencia de PM2.5 | Excluir | Fuga de información |
| Meteorología de la misma estación | Evaluar | Solo cerca del 30 % de las estaciones la mide |

```{admonition} Interpretación
:class: tip
Con estos predictores se entrena el modelo en el capítulo siguiente, {ref}`Modelo de predicción <modelo>`.
```